In [1]:
! pip install gradio


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from PIL import Image
import cv2

C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# Load model
model = load_model("best_MobileNetV2.keras")
IMG_SIZE = (160, 160)
best_threshold = 0.5

# Grad-CAM generation
def generate_gradcam(img_pil):
    try:
        img_resized = img_pil.resize(IMG_SIZE)
        img_array = tf.keras.preprocessing.image.img_to_array(img_resized) / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        # Rebuild base
        base_model = tf.keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights="imagenet")
        last_conv_layer = base_model.get_layer("out_relu")

        x = tf.keras.layers.GlobalAveragePooling2D()(last_conv_layer.output)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

        rebuilt_model = tf.keras.Model(inputs=base_model.input, outputs=output)
        rebuilt_model.set_weights(model.get_weights())

        grad_model = tf.keras.Model(inputs=rebuilt_model.input, outputs=[last_conv_layer.output, rebuilt_model.output])

        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_array)
            class_idx = tf.argmax(predictions[0])
            loss = predictions[:, class_idx]

        grads = tape.gradient(loss, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
        conv_outputs = conv_outputs[0]
        heatmap = tf.reduce_sum(pooled_grads * conv_outputs, axis=-1)
        heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap + tf.keras.backend.epsilon())
        heatmap = heatmap.numpy()

        heatmap = cv2.resize(heatmap, IMG_SIZE)
        heatmap = np.uint8(255 * heatmap)
        heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        superimposed_img = cv2.addWeighted(np.array(img_resized), 0.6, heatmap_color, 0.4, 0)
        return Image.fromarray(superimposed_img.astype(np.uint8))
    
    except Exception as e:
        print(f"[Grad-CAM ERROR]: {e}")
        return None

# Prediction
def predict_skin_cancer(img):
    if img is None:
        return ("<h4 style='margin: 0;'>Prediction Result:</h4><p style='color: #e67e22;'>⚠️ Please upload a skin lesion image.</p>",
                "<p>No data</p>", None)

    img_input = img.resize(IMG_SIZE)
    img_array = image.img_to_array(img_input) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prob = float(model.predict(img_array, verbose=0)[0][0])
    pred_label = "Malignant" if prob > best_threshold else "Benign"
    confidence = f"{prob * 100:.1f}%" if pred_label == "Malignant" else f"{(1 - prob) * 100:.1f}%"
    color = "#e74c3c" if pred_label == "Malignant" else "#2ecc71"
    diagnosis_html = f"<span style='color: {color}; font-weight: bold;'>{pred_label}</span>"

    desc = (
        "<br><br><b>Description:</b> Cancerous lesion. Can grow and spread. Requires immediate medical evaluation."
        if pred_label == "Malignant"
        else "<br><br><b>Description:</b> Non-cancerous lesion. Usually harmless and not life-threatening. However, monitoring is still recommended."
    )

    result_text = f"<h4 style='margin: 0;'>Prediction Result:</h4>Diagnosis: {diagnosis_html}<br>Confidence: {confidence}{desc}"
    benign_percent = (1 - prob) * 100
    malignant_percent = prob * 100

    bar_html = f"""
    <div style='margin-top:10px;'>
        <div style='font-weight: bold; margin-bottom: 2px;'>Benign: {benign_percent:.1f}%</div>
        <div style='background: #2ecc71; height: 10px; width: {benign_percent}%; margin-bottom: 10px;'></div>
        <div style='font-weight: bold; margin-bottom: 2px;'>Malignant: {malignant_percent:.1f}%</div>
        <div style='background: #e74c3c; height: 10px; width: {malignant_percent}%; margin-bottom: 10px;'></div>
    </div>
    """

    gradcam_img = generate_gradcam(img)
    return result_text, bar_html, gradcam_img

def clear_all():
    return None, "<h4 style='margin: 0;'>Prediction Result:</h4><p>Awaiting prediction...</p>", "<p>No data</p>", None

# GUI
with gr.Blocks(theme=gr.themes.Soft(), title="Skin Cancer Detector") as demo:
    gr.HTML("""
    <div style="text-align:center; margin-bottom: 1rem;">
        <h1>Skin Cancer Detector</h1>
        <h4 style="color: #555;">Automated Dermatological Screening Tool</h4>
    </div>
    """)

    gr.Markdown("""
    Upload a clear, well-lit image of a skin lesion (JPEG/PNG).  
    This tool will analyze the image and classify the lesion as **Benign** (green line) or **Malignant** (red line).  
    Press **Submit** to get the prediction and see the confidence score and Grad-CAM explanation.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="Upload Skin Lesion Image", interactive=True)
            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary")
                submit_btn = gr.Button("Submit", variant="primary")

        with gr.Column(scale=1):
            output_text = gr.HTML(label="Prediction Result", elem_id="prediction-result", value="<h4 style='margin: 0;'>Prediction Result:</h4><p>Awaiting prediction...</p>", show_label=True)
            with gr.Accordion("Prediction Score (Unfold to view full probabilities)", open=False):
                output_scores = gr.HTML(label="Probability Scores", value="<p>No data</p>")
            gradcam_output = gr.Image(label="Grad-CAM Heatmap", interactive=False)

    with gr.Accordion("Understanding Grad-CAM Heatmap", open=False):
        gr.Markdown("""
        The Grad-CAM heatmap shows **which parts of the image the AI model focused on** when making its prediction:

        - **Red / Orange**: Areas the model strongly focused on — considered **most important** for the diagnosis.  
        - **Yellow**: Moderately important regions.  
        - **Light Blue**: Less important — the model considered them but not strongly.  
        - **Dark Blue**: Ignored — **not relevant** to the prediction.

        #### What this means:
        - If **red/yellow appears over the lesion**, the model is attending to important features like **asymmetry, dark spots, texture, or borders**, which are common signs of malignant lesions.
        - If the heatmap focuses **outside the lesion**, the model might be distracted — possibly due to unclear images or poor lighting.
        - When the **center of the lesion** shows red/yellow, it suggests the model found high-risk features it learned from medical training data (like crusting or uneven color).

        >  This heatmap does **not diagnose**. It only shows **where the AI focused** when deciding between benign and malignant.
        """)

    with gr.Accordion("About this Model", open=False):
        gr.Markdown("""
        **Dataset:**  
        - HAM10000 (Skin Cancer MNIST) from Kaggle  
        - https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000  

        **Architecture & Training:**  
        - MobileNetV2 fine-tuned  
        - Input: 160×160 normalized images  
        - Techniques: Augmentation, balancing, Early stopping, model checkpoint  

        **Threshold:**  
        - 0.5 cutoff for binary classification
        """)

    gr.Markdown("""
    <div style="text-align:center; font-size:0.9rem; color:#777; margin-top: 1rem;">
        Areen Mohammed Meraj Zaki • Student ID: TP081741 •  
        Master of Science in Artificial Intelligence • Supervisor: Dr. Kuan Yik Junn  
    </div>
    """)

    submit_btn.click(fn=predict_skin_cancer, inputs=img_input, outputs=[output_text, output_scores, gradcam_output])
    clear_btn.click(fn=clear_all, inputs=None, outputs=[img_input, output_text, output_scores, gradcam_output])

demo.launch(share=True)


C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gradio\components\base.py:199: UserWarning: show_label has no effect when container is False.
  warnings.warn("show_label has no effect when container is False.")


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4693bdee94c15fb54c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
